In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col, expr, current_timestamp, to_timestamp, sha2, concat_ws, coalesce, monotonically_increasing_id
from delta.tables import DeltaTable
from pyspark.sql import Window
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file

storage_account = os.getenv('STORAGE_ACCOUNT')

# ADSL configuration
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope = "hospitalanalyticsvaultscope", key = "storage-connection")
)

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/patient_flow"
gold_dim_patient = f"abfss://gold@{storage_account}.dfs.core.windows.net/dim_patient"
gold_dim_department = f"abfss://gold@{storage_account}.dfs.core.windows.net/dim_department"
gold_fact = f"abfss://gold@{storage_account}.dfs.core.windows.net/fact_patient_flow"

# Read silver data (assume append-only)
silver_df = spark.read.format("delta").load(silver_path)

# Define window for latest admission per patient
w = Window.partitionBy("patient_id").orderBy(col("admission_time").desc())

silver_df = (
    silver_df
    .withColumn("row_num", F.row_number().over(w))  # Rank by latest admission_time
    .filter(F.col("row_num") == 1)                  # Keep only latest row
    .drop("row_num")
)

# Patient Dimension Table Creation
# Prepare incoming dimension records (deduplicated per patient, latest record)
incoming_patient = (silver_df
                    .select("patient_id", "gender", "age")
                    .withColumn("effective_from", current_timestamp())
                    )

# Create target if not exists
if not DeltaTable.isDeltaTable(spark, gold_dim_patient):
    # Initialize table with schema and empty data
    incoming_patient.withColumn("surrogate_key", F.monotonically_increasing_id()) \
                    .withColumn("effective_to", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("overwrite").save(gold_dim_patient)

# Load target as DeltaTable
target_patient = DeltaTable.forPath(spark, gold_dim_patient)

# Create an expression to detect attribute changes (hash or explicit comparisons)
# We'll use a simple concat hash to detect changes
incoming_patient = incoming_patient.withColumn(
    "_hash",
    F.sha2(F.concat_ws("||", F.coalesce(col("gender"), lit("NA")), F.coalesce(col("age").cast("string"), lit("NA"))), 256)
)

target_patient = spark.read.format("delta").load(gold_dim_patient).withColumn(
    "_target_hash",
    F.sha2(F.concat_ws("||", F.coalesce(col("gender"), lit("NA")), F.coalesce(col("age").cast("string"), lit("NA"))), 256)
).select("surrogate_key", "patient_id", "gender", "age", "is_current", "_target_hash", "effective_from", "effective_to")

# Create temp views for merge
incoming_patient.createOrReplaceTempView("incoming_patient_tmp")
target_patient.createOrReplaceTempView("target_patient_tmp")

# We'll implement in two steps using Delta MERGE (safe & explicit)

# 1) Mark old current rows as not current where changed
changes_df = spark.sql("""
SELECT t.surrogate_key, t.patient_id
FROM target_patient_tmp t
JOIN incoming_patient_tmp i
    ON t.patient_id = i.patient_id
WHERE t.is_current = true AND t._target_hash <> i._hash
""")

changed_keys = [row['surrogate_key'] for row in changes_df.collect()]

if changed_keys:
    # Update existing current records: set is_current=false and effective_to=current_timestamp()
    target_patient.update(
        condition = expr("is_current = true AND surrogate_key IN ({})".format(",".join([str(k) for k in changed_keys]))),
        set = {
            "is_current": expr("false"), 
            "effective_to": expr("current_timestamp()")
        }
    )


# 2) Insert new rows for changed & new records
# Build insert DF: join incoming with target to figure new inserts where either not exists or changed
inserts_df = spark.sql("""
SELECT i.patient_id, i.gender, i.age, i.effective_from, i._hash
FROM incoming_patient_tmp i
LEFT JOIN target_patient_tmp t
    ON i.patient_id = t.patient_id AND t.is_current = true
WHERE t.patient_id IS NULL OR t._target_hash <> i._hash                       
""").withColumn("surrogate_key", F.monotonically_increasing_id()) \
    .withColumn("effective_to", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True)) \
    .select("surrogate_key", "patient_id", "gender", "age", "effective_from", "effective_to", "is_current")

# Append new rows
if inserts_df.count() > 0:
    inserts_df.write.format("delta").mode("append").save(gold_dim_patient)

# Department Dimension Table Creation

# Prepare incoming (latest per patient feed snapshot)
incoming_dept = (silver_df
                 .select("department", "hospital_id")
                )

# Add hash and dedupe incoming (one row per natural key)
incoming_dept = incoming_dept.dropDuplicates(["department","hospital_id"]) \
    .withColumn("surrogate_key", monotonically_increasing_id())

# Initalizae table if missing
incoming_dept.select("surrogate_key", "department", "hospital_id") \
    .write.format("delta").mode("overwrite").save(gold_dim_department)

# Create Fact table

# Read current dims (filter is_current=true)
dim_patient_df = (spark.read.format("delta").load(gold_dim_patient)
                  .filter(col("is_current") == True)
                  .select(col("surrogate_key").alias("surrogate_key_patient"), "patient_id", "gender", "age"))

dim_dept_df = (spark.read.format("delta").load(gold_dim_department)
                    .select(col("surrogate_key").alias("surrogate_key_department"), "department", "hospital_id"))

# Build base fact from silver evens
fact_base = (silver_df
             .select("patient_id", "department", "hospital_id", "admission_time", "discharge_time", "bed_id")
             .withColumn("admission_date", F.to_date("admission_time"))
             )

# Join to get surrogate keys
fact_enriched = (fact_base
                 .join(dim_patient_df, on="patient_id", how="left")
                 .join(dim_dept_df, on=["department", "hospital_id"], how="left")
                )

# Compute metrics
fact_enriched = fact_enriched.withColumn("lenght_of_stay_hours",
                                         (F.unix_timestamp(col("discharge_time")) - F.unix_timestamp(col("admission_time"))) / 3600.0) \
                             .withColumn("is_currently_admitted", F.when(col("discharge_time") > current_timestamp(), lit(True)).otherwise(lit(False))) \
                            .withColumn("event_ingesion_time", current_timestamp())


# Let's make column names explicit instead:
fact_final = fact_enriched.select(
    monotonically_increasing_id().alias("fact_id"),
    col("surrogate_key_patient").alias("patient_sk"),
    col("surrogate_key_department").alias("department_sk"),
    "event_ingesion_time",
    "discharge_time",
    "length_of_stay_hours",
    "is_currently_admitted",
    "bed_id",
    "event_ingestion_time"
)

# Persist fact table partitioned by admission_date (helps Synapse / queries)
fact_final.write.format("delta").mode("overwrite").save(gold_fact)

# Quick sanity checks
print("Patient dim count:", spark.read.format("delta").load(gold_dim_patient).count())
print("Department dim count:", spark.read.format("delta").load(gold_dim_department).count())
print("Fact rows:", spark.read.format("delta").load(gold_fact).count())


Patient dim count: 1819
Department dim count: 49
Fact rows: 1819


In [0]:
display(spark.read.format("delta").load(gold_dim_patient))

patient_id,gender,age,effective_from,surrogate_key,effective_to,is_current
000fb4b3-fd72-4177-abd1-6f907a18f7cf,Female,65,2026-05-30T04:55:33.070715Z,0,null,true
002003e9-1c91-49ef-a70b-de1c14c65d5d,Female,90,2026-05-30T04:55:33.070715Z,1,null,true
0084ff62-22d7-4956-a287-71009d0ecb9e,Male,6,2026-05-30T04:55:33.070715Z,2,null,true
00f23048-9348-4ff6-a444-d7e949cfcf81,Female,19,2026-05-30T04:55:33.070715Z,3,null,true
0138c5af-269a-436e-a93a-f72955a9a7b0,Male,11,2026-05-30T04:55:33.070715Z,4,null,true
015ccf53-6afe-4d89-a73a-32be3ed0508a,Female,36,2026-05-30T04:55:33.070715Z,5,null,true
0189b5cd-4aa6-4c82-8668-c47afc2d2803,Male,60,2026-05-30T04:55:33.070715Z,6,null,true
01b7a439-f00c-4567-a973-c255d284eb51,Male,27,2026-05-30T04:55:33.070715Z,7,null,true
01dbc900-538d-4d89-8a64-c85c5a08bcb6,Female,84,2026-05-30T04:55:33.070715Z,8,null,true
022970ad-b64e-451e-8429-6d3825753a89,Male,26,2026-05-30T04:55:33.070715Z,9,null,true


In [0]:
display(spark.read.format("delta").load(gold_dim_department))

surrogate_key,department,hospital_id
0,Pediatrics,5
1,Surgery,3
2,Surgery,2
3,ICU,3
4,Emergency,5
5,Emergency,7
6,Cardiology,1
7,Surgery,1
8,Oncology,3
9,Emergency,6


In [0]:
display(spark.read.format("delta").load(gold_fact))

fact_id,patient_sk,department_sk,admission_time,discharge_time,lenght_of_stay_hours,is_currently_admitted,bed_id,event_ingesion_time
0,0,42,2026-05-26T15:20:06.951397Z,2026-05-28T04:20:06.951397Z,37.0,false,91,2026-05-30T04:55:42.43895Z
1,1,28,2026-05-27T02:16:58.732365Z,2026-05-29T10:16:58.732365Z,56.0,false,462,2026-05-30T04:55:42.43895Z
2,2,42,2026-05-27T04:12:50.435871Z,2026-05-28T11:12:50.435871Z,31.0,false,455,2026-05-30T04:55:42.43895Z
3,3,29,2026-05-29T02:08:26.138104Z,2026-05-31T22:08:26.138104Z,68.0,true,227,2026-05-30T04:55:42.43895Z
4,4,3,2026-05-27T10:55:26.214662Z,2026-05-29T23:55:26.214662Z,61.0,false,413,2026-05-30T04:55:42.43895Z
5,5,15,2026-05-28T17:10:12.2528Z,2026-05-31T04:10:12.2528Z,59.0,true,90,2026-05-30T04:55:42.43895Z
6,6,15,2026-05-29T10:21:58.083399Z,2026-06-01T07:21:58.083399Z,69.0,true,20,2026-05-30T04:55:42.43895Z
7,7,41,2026-05-29T11:23:56.01Z,2026-05-29T13:05:17.92808Z,1.6891666666666667,false,199,2026-05-30T04:55:42.43895Z
8,8,48,2026-05-29T06:52:50.028703Z,2026-05-31T15:52:50.028703Z,57.0,true,199,2026-05-30T04:55:42.43895Z
9,9,39,2026-05-28T17:13:32.488665Z,2026-05-29T09:13:32.488665Z,16.0,false,173,2026-05-30T04:55:42.43895Z
